In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Importing the Dependencies

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


In [3]:
# loading the dataset to a Pandas Dataframe
credit_card_data = pd.read_csv('/content/credit_card_fraud_10k.csv')

In [4]:
credit_card_data.head()

,transaction_id,amount,transaction_hour,merchant_category,foreign_transaction,location_mismatch,device_trust_score,velocity_last_24h,cardholder_age,is_fraud
0,1,84.47,22,Electronics,0,0,66,3,40,0
1,2,541.82,3,Travel,1,0,87,1,64,0
2,3,237.01,17,Grocery,0,0,49,1,61,0
3,4,164.33,4,Grocery,0,1,72,3,34,0
4,5,30.53,15,Food,0,0,79,0,44,0


In [5]:
credit_card_data.tail()

,transaction_id,amount,transaction_hour,merchant_category,foreign_transaction,location_mismatch,device_trust_score,velocity_last_24h,cardholder_age,is_fraud
9995,9996,350.91,22,Food,0,0,99,4,37,0
9996,9997,410.04,5,Clothing,0,0,70,3,25,0
9997,9998,527.75,21,Electronics,0,0,44,2,45,0
9998,9999,91.20,2,Electronics,0,0,38,0,37,0
9999,10000,44.06,2,Clothing,0,0,38,0,66,0


In [6]:
credit_card_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   transaction_id       10000 non-null  int64  
 1   amount               10000 non-null  float64
 2   transaction_hour     10000 non-null  int64  
 3   merchant_category    10000 non-null  object 
 4   foreign_transaction  10000 non-null  int64  
 5   location_mismatch    10000 non-null  int64  
 6   device_trust_score   10000 non-null  int64  
 7   velocity_last_24h    10000 non-null  int64  
 8   cardholder_age       10000 non-null  int64  
 9   is_fraud             10000 non-null  int64  
dtypes: float64(1), int64(8), object(1)
memory usage: 781.4+ KB


In [7]:
credit_card_data.isnull().sum()

,0
transaction_id,0
amount,0
transaction_hour,0
merchant_category,0
foreign_transaction,0
location_mismatch,0
device_trust_score,0
velocity_last_24h,0
cardholder_age,0
is_fraud,0


In [8]:

credit_card_data['is_fraud'].value_counts()

,count
is_fraud,
0,9849
1,151


In [9]:
legit = credit_card_data[credit_card_data['is_fraud'] == 0]
fraud = credit_card_data[credit_card_data['is_fraud'] == 1]

In [10]:
print(legit.shape)
print(fraud.shape)

(9849, 10)
(151, 10)


In [11]:
legit.amount.describe()

,amount
count,9849.000000
mean,175.333015
std,173.986837
min,0.000000
25%,50.990000
50%,122.110000
75%,241.650000
max,1471.040000


In [12]:
fraud.amount.describe()

,amount
count,151.000000
mean,216.182980
std,248.120467
min,0.110000
25%,41.530000
50%,118.940000
75%,341.695000
max,1185.070000


In [13]:
credit_card_data.groupby('is_fraud').mean(numeric_only=True)

,transaction_id,amount,transaction_hour,foreign_transaction,location_mismatch,device_trust_score,velocity_last_24h,cardholder_age
is_fraud,,,,,,,,
0,5004.129861,175.333015,11.712154,0.090974,0.079704,62.165804,1.990557,43.469794
1,4763.741722,216.182980,3.841060,0.543046,0.476821,37.867550,3.205298,43.397351


In [14]:
legit_sample = legit.sample(n=492)

In [15]:
new_dataset = pd.concat([legit_sample, fraud], axis=0)

In [16]:
new_dataset.head()

,transaction_id,amount,transaction_hour,merchant_category,foreign_transaction,location_mismatch,device_trust_score,velocity_last_24h,cardholder_age,is_fraud
4156,4157,139.89,7,Travel,0,0,77,1,59,0
5832,5833,304.76,7,Food,0,0,37,2,57,0
9983,9984,433.08,20,Grocery,0,0,83,1,44,0
9766,9767,180.95,22,Clothing,0,1,49,2,32,0
831,832,108.41,15,Electronics,0,0,35,3,25,0


In [17]:
new_dataset.tail()

,transaction_id,amount,transaction_hour,merchant_category,foreign_transaction,location_mismatch,device_trust_score,velocity_last_24h,cardholder_age,is_fraud
9526,9527,398.66,3,Grocery,0,0,30,5,61,1
9553,9554,182.65,2,Electronics,1,1,31,2,28,1
9587,9588,93.93,1,Travel,0,1,27,3,69,1
9779,9780,172.00,2,Grocery,0,1,56,5,24,1
9973,9974,11.22,1,Grocery,0,1,25,4,32,1


In [18]:
new_dataset['is_fraud'].value_counts()

,count
is_fraud,
0,492
1,151


In [19]:
new_dataset.groupby('is_fraud').mean(numeric_only=True)

,transaction_id,amount,transaction_hour,foreign_transaction,location_mismatch,device_trust_score,velocity_last_24h,cardholder_age
is_fraud,,,,,,,,
0,4956.386179,180.47689,11.79878,0.103659,0.060976,61.892276,2.004065,43.861789
1,4763.741722,216.18298,3.84106,0.543046,0.476821,37.867550,3.205298,43.397351


In [20]:
X = new_dataset.drop(columns=['is_fraud', 'transaction_id'], axis=1)
Y = new_dataset['is_fraud']

X = pd.get_dummies(X, columns=['merchant_category'], drop_first=True)

In [21]:
print(X)

      amount  transaction_hour  foreign_transaction  location_mismatch  \
4156  139.89                 7                    0                  0   
5832  304.76                 7                    0                  0   
9983  433.08                20                    0                  0   
9766  180.95                22                    0                  1   
831   108.41                15                    0                  0   
...      ...               ...                  ...                ...   
9526  398.66                 3                    0                  0   
9553  182.65                 2                    1                  1   
9587   93.93                 1                    0                  1   
9779  172.00                 2                    0                  1   
9973   11.22                 1                    0                  1   

      device_trust_score  velocity_last_24h  cardholder_age  \
4156                  77                  1     

In [22]:
print(Y)

4156    0
5832    0
9983    0
9766    0
831     0
       ..
9526    1
9553    1
9587    1
9779    1
9973    1
Name: is_fraud, Length: 643, dtype: int64


In [23]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, stratify=Y, random_state=2)

In [24]:
print(X.shape, X_train.shape, X_test.shape)

(643, 11) (514, 11) (129, 11)


In [25]:
model = LogisticRegression(max_iter=1000)

In [26]:
model.fit(X_train, Y_train)

LogisticRegression(max_iter=1000)

In [27]:
# Accuracy on training data
X_train_prediction = model.predict(X_train)
training_data_accuracy = accuracy_score(Y_train, X_train_prediction)
print('Accuracy on training data:', training_data_accuracy)

Accuracy on training data: 0.9669260700389105


In [28]:
# Accuracy on test data
X_test_prediction = model.predict(X_test)
test_data_accuracy = accuracy_score(Y_test, X_test_prediction)
print('Accuracy on test data:', test_data_accuracy)

Accuracy on test data: 0.937984496124031
